In [1]:
import os, shutil, json

In [2]:
# config defs

config_obj = {
    'Main-Path':"/home/$user/Documents/input", # defines where input begins 
    'Storage-Defs':{
        'images':'/home/$user/images/', # define where things get stored 
                                                 # can be narrow or broad
        'MULTI-PATH':{
            'BASE_PATH':'/home/$user/Documents/', # defines a base path
            'DEF_NAME':'dmv',
            'docs':'Docs', # now becomes /home/$user/Documents/Docs
            'music':'Music',
            'video':'Video',
        },
        'school':'/home/$user/School/',
    },
    'Defs':{
        'images':{
            'group-by':['name','type'], # groups file(name) and/or file(type) and/or file(ext)
                                        # into folders type is a higher priority than name 
                                        # eg file bird.png -> image/bird
                                        # folders are based on highest common name
                                        # eg. many files are named bird
                                        # with a break(_-.~|\ )<-'\ ' is a space 
                                        # then a folder called bird will be made
            'exclude-begins-with':['img'], # exclude files begin with this from creating a folder 
                                           # does not exclude them from being sorted 
                                           # only stops in this case from creating a folder named img
            'exclude-ends-with':['img'],   # opposite of beings with
            'exclude-contains':['img'],    # exclude is this is in the name
            'exclude-contains-break':True, # helps contains will force the check to be only catch 
                                           # files that are named with a break
                                           # eg finds img 102, 102 img, 1 img 02 but will not fild lolimg1234
            'exclude-file':['lolimg1234'],  # exclude specific files from being grouped
            'reverse-lookup':True, # looks up the word in the name
                                   # (default is to check only the first word)
                                   # moving everything based on highest valued name
                                   # eg Bird Raven, egal Bird
                                   # these both have birth in their name
                                   # they belong to a folder named bird/
            'waterfall':False, # makes a folder for every multiple category
                               # eg files Bird egal, bird egal 2, bird raven
                               # there will be a folder called bird/ 
                               # and another nested called egal/
            'waterfall-priority-order':False, # this provides priority to names with most value 
                                              # eg files Bird egal, bird egal 2, bird raven
                                              # two outputs egal/bird/ bird/
            'waterfall-priority-merge':False, # this merges nested directories for
                                              # waterfall-priority-order 
                                              # eg Bird egal, bird egal 2, bird raven 
                                              # two outputs bird-egal/ bird/
            'ext-defs':{
                # and ext-defs that trumps the overhead one eg.
                'web-ready':['png','jpeg'],
                'high-quality':['heic','raw']
                # right now jpg files will be the only file that gets sorted
                # into a folder called image (based on the below ext-defs)
            },
        },
        'dmv/docs':{
            'group-by':['name','type'],
            'ext-defs':{
                'pdf':['pdf'], 
                'docs':'text' # defines a alias
            },
            'ext-type':['docs','sheet','pdf'], # defines the ext-types that can be sorted into this folder
            'exclude-path':False # searches everywhere from any known location eg Main-Path 
                                 # and anything defined in Storage-Defs
                                 # Default is to only search Main-Path
        },
        'dmv/music':{
            'group-by':['name'],
            'ext-type':['audio'],
            'exclude-path':True # searches only inside of the directory $dmv['BASE-PATH']+$dmv['music']
        },
        'dmv/video':{
            'group-by':['name'],
            'ext-type':['video'],
            'exclude-path':['images','dmv'], # dmv here excludes $dmv['BASE-PATH']+*
            'include-path':['dmv/video'] # defines an allowed path which has higher priority than the exclude-path
        },
        'school':{
            'group-by':['name','type'],
            'reverse-lookup':True,
            'waterfall':True,
            'exclude-begins-with':['Joe Schmo'],
            'exclude-contains-break':True,
            'tag':['school','project','class','econ'] # files tagged with these will end up in this folder
                                                      # files without these tags are skipped 
                                                      # if the folder contains the file without one of the tags 
                                                      # the file will remain there and be sorted like any other file 
                                                      # the tag def in any on of these does 
                                                      # enforce a no look up policy so that other folders cannot 
                                                      # remove a file from these folders
            #'enforce-tag-policy':False # default is True
                                        # when this is false
                                        # files without these tags may be resorted somewhere else 
                                        # this may also be defined to [low|medium|high] 
                                        # which stages how much to enforce 
            # low - any file without a tag may be moved if it doesn't have a matched name 
                                                            # (eg exclude-[begins-with|ends-with|contains] definition 
                                                            # defines a matchable name
                                                            # or use 'match-name'
            # medium - any file not in a child directory of the parent/def may be moved
            # high - any file without a tag may be reviewed
        },
        'default':{ # gets applied to anything that is not defined above
            'group-by':['name','type'],
            'reverse-lookup':False,
            'waterfall':False,
        }
    },
    'ext-defs':{
        # file extension class. this defines group-by type folder named
        # format:
        # classname: [ext(without the \.),...]
        'image':['png','jpg','jpeg','heic'],
        'video':['mp4','avi','mkv','mpg','mpeg','avi','mkv'],
        'audio':['mp3','wav','ogg','flac','wav','aiff','ogg'],
        'presentation':['ppt','pptx'],
        'text':['txt','odt','docx'],
        'sheet':['xls','xlsx','csv'],
        'database':['sqlite','sqlite3'],
        'webpage':['htm','html','htm','htm'],
        'stylesheet':['css','css'],
        'program':['py','pyc','js'],
        'executable':['exe','dmg','sh','bat','jar'],
    }
}